In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ====================== LOAD DATA ======================

df = pd.read_csv("HR_Analytics.csv")

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)

# ====================== SQL DATABASE ======================

conn = sqlite3.connect("hr_analytics.db")

df.to_sql(
    "employees",
    conn,
    index=False,
    if_exists="replace"
)

print("SQLite Database Created!")

# ====================== SQL ANALYSIS ======================

print("\n=== Attrition by Department ===")

query1 = """
SELECT Department,
       COUNT(*) AS Total,
       SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS Attrition_Count
FROM employees
GROUP BY Department
"""

print(pd.read_sql_query(query1, conn))

print("\n=== Attrition by Age Group ===")

query2 = """
SELECT AgeGroup,
       COUNT(*) AS Total,
       SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS Attrition_Count
FROM employees
GROUP BY AgeGroup
"""

print(pd.read_sql_query(query2, conn))

# ====================== VISUALIZATION 1 ======================

attrition_dept = (
    df.groupby("Department")["Attrition"]
    .apply(lambda x: (x == "Yes").mean() * 100)
)

plt.figure(figsize=(8,5))
attrition_dept.plot(kind="bar")
plt.title("Attrition Rate by Department")
plt.ylabel("Attrition %")
plt.tight_layout()
plt.show()

# ====================== VISUALIZATION 2 ======================

attrition_age = (
    df.groupby("AgeGroup")["Attrition"]
    .apply(lambda x: (x == "Yes").mean() * 100)
)

plt.figure(figsize=(8,5))
attrition_age.plot(kind="bar")
plt.title("Attrition Rate by Age Group")
plt.ylabel("Attrition %")
plt.tight_layout()
plt.show()

# ====================== VISUALIZATION 3 ======================

plt.figure(figsize=(8,5))

plt.hist(
    df[df["Attrition"]=="No"]["MonthlyIncome"],
    bins=20,
    alpha=0.7,
    label="Stayed"
)

plt.hist(
    df[df["Attrition"]=="Yes"]["MonthlyIncome"],
    bins=20,
    alpha=0.7,
    label="Left"
)

plt.legend()
plt.title("Monthly Income Distribution")
plt.xlabel("Monthly Income")
plt.ylabel("Count")
plt.show()

# ====================== DATA PREPROCESSING ======================

df_model = df.copy()

# Target Encoding
df_model["Attrition"] = df_model["Attrition"].map(
    {"Yes":1, "No":0}
)

# Categorical Columns
cat_cols = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime",
    "SalarySlab",
    "AgeGroup"
]

# One Hot Encoding
df_model = pd.get_dummies(
    df_model,
    columns=cat_cols,
    drop_first=True
)

# Remove unwanted columns
drop_cols = ["Attrition"]

for col in ["EmpID", "EmployeeNumber", "Over18"]:
    if col in df_model.columns:
        drop_cols.append(col)

X = df_model.drop(columns=drop_cols)

y = df_model["Attrition"]

# Check object columns
print("\nRemaining Object Columns:")
print(X.select_dtypes(include="object").columns.tolist())

# If still object columns exist, remove them
X = X.select_dtypes(exclude="object")

# ====================== TRAIN TEST SPLIT ======================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ====================== MODEL BUILDING ======================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

# ====================== PREDICTION ======================

pred = model.predict(X_test)

print("\n=== Classification Report ===")
print(classification_report(y_test, pred))

# ====================== FEATURE IMPORTANCE ======================

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

importance = importance.sort_values(
    ascending=False
)

print("\n=== Top 10 Important Features ===")
print(importance.head(10))

# ====================== SAVE RESULTS ======================

importance.head(10).to_csv(
    "Top10_Feature_Importance.csv"
)

conn.close()

print("\n✅ HR Analytics Project Completed Successfully!")